# Drug liver toxicity starter notebook

This notebook uses the SQL-backed cohort builder in `drug_liver_toxicity/cohort.py`, which resolves OMOP concept IDs from names and executes the cohort logic against BigQuery/OMOP tables.

## Workflow
1. Connect to the ResearchAllOfUs OMOP dataset.
2. Resolve concept IDs for the target drug, liver disease exclusions, and lab concepts.
3. Build the drug exposure cohort.
4. Define the liver toxicity phenotype.
5. Run a simple baseline model.

> Replace the placeholder project and dataset values below with your actual ResearchAllOfUs BigQuery project and OMOP dataset.


In [ ]:
from google.cloud import bigquery

from drug_liver_toxicity.config import CohortConfig
from drug_liver_toxicity.cohort import build_drug_cohort
from drug_liver_toxicity.phenotype import define_liver_toxicity_outcome
from drug_liver_toxicity.analysis import run_baseline_model

print("Starter notebook loaded successfully.")

# Example BigQuery connection for ResearchAllOfUs OMOP data.
# Replace with your actual project and dataset.
PROJECT_ID = "wb-silky-artichoke-2408"
DATASET_ID = "C2025Q4R6"

# --- Dataset that contains t_ent_person table with properly
# --- populated column indicating which persons have genomics data
INDEX_DATASET = "wb-silky-artichoke-2408.C2025Q4R6_index_061026"

PROJECT_DATASET = "wb-spotless-beet-1982.claudiof_ds1"

client = bigquery.Client(project=PROJECT_ID)

cohort_config = CohortConfig(
    drug_name="duloxetine",
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    drug_exposure_table="drug_exposure",
    condition_occurrence_table="condition_occurrence",
    measurement_table="measurement",
    person_table="person",
    concept_table="concept",
    concept_ancestor_table="concept_ancestor",
    index_dataset=INDEX_DATASET,
    project_dataset=PROJECT_DATASET
)

# Example usage:
# cohort = build_drug_cohort(conn=client, config=cohort_config)


In [ ]:
# This notebook is now designed for BigQuery OMOP tables.
# The synthetic DataFrame example is no longer the default workflow.
# Instead, connect to your ResearchAllOfUs dataset and call the cohort builder
# with project_id and dataset_id.

print("BigQuery OMOP workflow is ready.")


In [ ]:
# Build the cohort against a ResearchAllOfUs OMOP BigQuery dataset using config.
cohort = build_drug_cohort(
    conn=client,
    config=cohort_config,
)

print(cohort.head())
print(f"Cohort size: {len(cohort)}")


In [ ]:
# Define the outcome from the SQL-built cohort.
# This notebook expects the cohort to already contain person_id and index_date.
outcome_cohort = define_liver_toxicity_outcome(
    cohort=cohort,
    measurement=client.query(
        f"""
        SELECT person_id, measurement_datetime, measurement_concept_id, value_as_number
        FROM `{PROJECT_ID}.{DATASET_ID}.measurement`
        WHERE person_id IN (SELECT DISTINCT person_id FROM `{PROJECT_ID}.{DATASET_ID}.drug_exposure`)
        """
    ).to_dataframe(),
    outcome_window_days=365,
    strict_mode=False,
)

print(outcome_cohort[["person_id", "index_date", "liver_toxicity", "outcome_date"]].head())
print(outcome_cohort["liver_toxicity"].value_counts(dropna=False).to_dict())


In [ ]:
# Run a simple baseline model on the outcome cohort.
covariates = [
    "year_of_birth",
    "sex_at_birth",
    "race",
    "ethnicity",
]

metrics = run_baseline_model(
    cohort=outcome_cohort,
    covariates=covariates,
)

print("AUC:", metrics["auc"])
print("Average Precision:", metrics["average_precision"])


## Next steps

- Replace the placeholder BigQuery project and dataset with your actual ResearchAllOfUs values.
- Set `STUDY.drug_name` and/or concept mappings for each drug you analyze.
- Add genotype/protein merging in a dedicated module for later analysis.
- Convert this workflow into a more formal analysis pipeline once concept mapping is finalized.
